<a href="https://colab.research.google.com/github/blackwatermelon0000/NLP/blob/main/FA23_BAI_010_AyeshaMehmood.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AIC365 Natural Language Processing
**COMSATS University Islamabad, Main Campus - Department of Computer Science**



---
# Question 1: Customer Review Mining for an Online Electronics Store

TechMart wants to analyse customer feedback before building a sentiment system. The web team saved one product page as HTML, and the raw reviews must first be collected into a clean, structured form.

**Libraries allowed:** `bs4`, `pandas`, `collections`

## Q1 (a) Data Acquisition
**Task:** Store the HTML in a variable named `html_doc`. Using BeautifulSoup, extract the **product name**, **rating** (as an integer) and **review text** of every review into a list of dictionaries. A review with a missing rating must not crash the program; store its rating as `None`.

**Approach:**
- Each review is a `<div class="review">`, and the product name is its `data-product` attribute.
- `find()` returns `None` when the rating `<span>` is missing, so we check for that before converting to `int`.

In [ ]:
from bs4 import BeautifulSoup
from collections import Counter
import pandas as pd

# storing the given HTML in a variable
html_doc = """
<div class="review" data-product="Wireless Earbuds X2">
  <span class="rating">5</span>
  <p class="text">Superb sound quality!! Battery lasts all day.</p></div>
<div class="review" data-product="Wireless Earbuds X2">
  <span class="rating">2</span>
  <p class="text">Left earbud stopped working after a week.</p></div>
<div class="review" data-product="Smart Watch S5">
  <span class="rating">4</span>
  <p class="text">Good display, strap could be better.</p></div>
<div class="review" data-product="Smart Watch S5">
  <p class="text">Delivery was late but the watch is nice.</p></div>
<div class="review" data-product="Wireless Earbuds X2">
  <span class="rating">4</span>
  <p class="text">Value for money. Delivery was quick.</p></div>
<div class="review" data-product="Smart Watch S5">
  <span class="rating">1</span>
  <p class="text">Screen cracked on arrival. Very disappointed.</p></div>
"""

# parse the HTML so BeautifulSoup can search it
soup = BeautifulSoup(html_doc, "html.parser")

reviews = []  # list of dictionaries, one per review

# every review sits inside a <div class="review">
for div in soup.find_all("div", class_="review"):
    product = div["data-product"]                 # product name is an attribute of the div
    rating_tag = div.find("span", class_="rating")
    text = div.find("p", class_="text").text      # review text

    # if the rating tag is missing, find() gives None, so we store None (no crash)
    if rating_tag is None:
        rating = None
    else:
        rating = int(rating_tag.text)             # convert "5" to 5

    reviews.append({"product": product, "rating": rating, "text": text})

print("Extracted reviews:")
for r in reviews:
    print(r)

Extracted reviews:
{'product': 'Wireless Earbuds X2', 'rating': 5, 'text': 'Superb sound quality!! Battery lasts all day.'}
{'product': 'Wireless Earbuds X2', 'rating': 2, 'text': 'Left earbud stopped working after a week.'}
{'product': 'Smart Watch S5', 'rating': 4, 'text': 'Good display, strap could be better.'}
{'product': 'Smart Watch S5', 'rating': None, 'text': 'Delivery was late but the watch is nice.'}
{'product': 'Wireless Earbuds X2', 'rating': 4, 'text': 'Value for money. Delivery was quick.'}
{'product': 'Smart Watch S5', 'rating': 1, 'text': 'Screen cracked on arrival. Very disappointed.'}


## Q1 (b) Python Processing
**Task:** Using the list from part (a):
1. count the number of reviews per product,
2. compute the average rating per product (ignore `None` values),
3. print all review texts that mention the word "delivery" (case-insensitive).

**(i) Number of reviews per product**  (uses `collections.Counter`)

In [ ]:
# (i) number of reviews per product
product_names = [r["product"] for r in reviews]
counts = Counter(product_names)
print("\nReviews per product:")
for product, c in counts.items():
    print(product, "->", c)


Reviews per product:
Wireless Earbuds X2 -> 3
Smart Watch S5 -> 3


**(ii) Average rating per product**  (rows with `None` are skipped)

In [ ]:
# (ii) average rating per product (ignoring None)
ratings_by_product = {}                           # product -> list of its ratings
for r in reviews:
    if r["rating"] is not None:                   # skip missing ratings
        ratings_by_product.setdefault(r["product"], []).append(r["rating"])

print("\nAverage rating per product:")
for product, rating_list in ratings_by_product.items():
    avg = sum(rating_list) / len(rating_list)     # average = sum / count
    print(product, "->", round(avg, 2))


Average rating per product:
Wireless Earbuds X2 -> 3.67
Smart Watch S5 -> 2.5


**(iii) Reviews that mention "delivery"**  (`.lower()` makes the search case-insensitive)

In [ ]:
# (iii) reviews that mention "delivery" (lower() makes it case-insensitive)
print("\nReviews mentioning 'delivery':")
for r in reviews:
    if "delivery" in r["text"].lower():
        print("-", r["text"])


Reviews mentioning 'delivery':
- Delivery was late but the watch is nice.
- Value for money. Delivery was quick.


## Q1 (c) Saving the Data
**Task:** Convert the data into a pandas DataFrame, drop the rows with a missing rating, save it as `reviews_clean.csv` (without the index), then reload the file and print its shape and first rows.

**Note:** `None` turns the rating column into floats (5.0), so we convert it back to `int` before saving.

In [ ]:
df = pd.DataFrame(reviews)                        # list of dicts -> table

df = df.dropna(subset=["rating"])                 # drop rows with missing rating
df["rating"] = df["rating"].astype(int)           # None made ratings float (5.0), so convert back to int

df.to_csv("reviews_clean.csv", index=False)       # save without the index column

# reload the saved file to check it
df_loaded = pd.read_csv("reviews_clean.csv")
print("\nShape:", df_loaded.shape)
print(df_loaded.head())


Shape: (5, 3)
               product  rating                                           text
0  Wireless Earbuds X2       5  Superb sound quality!! Battery lasts all day.
1  Wireless Earbuds X2       2      Left earbud stopped working after a week.
2       Smart Watch S5       4           Good display, strap could be better.
3  Wireless Earbuds X2       4           Value for money. Delivery was quick.
4       Smart Watch S5       1  Screen cracked on arrival. Very disappointed.


---
# Question 2: Complaint Analysis for a Telecom Customer-Care Centre

A telecom company receives thousands of social-media complaints every day. Before any model can be trained, the raw text must be mined and cleaned. Two sample complaints are used throughout this question.

**Note:** NLTK data (`punkt`, `stopwords`, `wordnet`) is assumed to be downloaded. The download lines below are only needed in Colab.

### Setup: imports, NLTK data and the sample complaints

In [ ]:
import re
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.stem import PorterStemmer, WordNetLemmatizer

# download NLTK data (needed only in Colab; the exam says it is already downloaded)
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)
nltk.download("stopwords", quiet=True)
nltk.download("wordnet", quiet=True)
nltk.download("omw-1.4", quiet=True)

complaints = [
    "@TelcoCare my 4G internet is NOT working since 2 days!!! Call me on 0300-1234567 "
    "or mail ali.raza@gmail.com. #NetworkDown https://t.co/Xy12Ab",
    "Dr. Sana here. The bills were wrongly charged in Lahore. Nobody is listening to "
    "customers! Contact: 0321-7654321 #BillingIssue"
]

## Q2 (a) Information Extraction using Regex
**Task:** Write regular expressions to extract, from each complaint: Pakistani mobile numbers (format `03XX-XXXXXXX`), email addresses, hashtags, URLs and @mentions. Print the results for each complaint as a dictionary.

| Item | Pattern idea |
|---|---|
| Phone | `03` + 2 digits + `-` + 7 digits |
| Email | name `@` domain `.` extension |
| Hashtag | `#` followed by word characters |
| URL | `http(s)://` up to the next space |
| @mention | `@word`, using `\B` so the `@` inside an email is not matched |

In [ ]:
phone_pat   = r'03\d{2}-\d{7}'                  # 03 + 2 digits, dash, 7 digits
email_pat   = r'[\w.-]+@[\w-]+\.[a-zA-Z]{2,}'   # name @ domain . extension
hashtag_pat = r'#\w+'                           # # followed by word characters
url_pat     = r'https?://\S+'                   # http(s):// then everything until a space
mention_pat = r'\B@\w+'                         # \B stops it matching the @ inside an email

for i, text in enumerate(complaints, 1):
    result = {
        "phones":   re.findall(phone_pat, text),
        "emails":   re.findall(email_pat, text),
        "hashtags": re.findall(hashtag_pat, text),
        "urls":     re.findall(url_pat, text),
        "mentions": re.findall(mention_pat, text),
    }
    print(f"Complaint {i}:", result)

Complaint 1: {'phones': ['0300-1234567'], 'emails': ['ali.raza@gmail.com'], 'hashtags': ['#NetworkDown'], 'urls': ['https://t.co/Xy12Ab'], 'mentions': ['@TelcoCare']}
Complaint 2: {'phones': ['0321-7654321'], 'emails': [], 'hashtags': ['#BillingIssue'], 'urls': [], 'mentions': []}


## Q2 (b) Text Cleaning Pipeline
**Task:** Write a function `preprocess(text)` that does: sentence segmentation, removes URLs, mentions, emails and phone numbers, keeps hashtag words without the `#`, lowercases, removes punctuation and digits, tokenizes, and removes English stopwords. Apply it to both complaints and print the tokens of every sentence.

**Pipeline order:** sentence split, remove URL / email / mention / phone, drop `#`, lowercase, remove punctuation and digits, tokenize, remove stopwords.

In [ ]:
# Stopword handling: we REMOVE normal stopwords but KEEP negation words
# (not, no, nor). Removing "NOT" would turn "internet is NOT working"
# into "internet working", which flips the meaning of the complaint.
stop_words = set(stopwords.words("english")) - {"not", "no", "nor"}

def preprocess(text):
    all_sentences = []
    for sent in sent_tokenize(text):             # 1. sentence segmentation
        sent = re.sub(url_pat, "", sent)         # 2. remove URLs
        sent = re.sub(email_pat, "", sent)       #    remove emails
        sent = re.sub(mention_pat, "", sent)     #    remove @mentions
        sent = re.sub(phone_pat, "", sent)       #    remove phone numbers
        sent = sent.replace("#", "")             # 3. keep hashtag word, drop only the #
        sent = sent.lower()                      # 4. lowercase
        sent = re.sub(r'[^a-z\s]', "", sent)     # 5. remove punctuation and digits
        tokens = word_tokenize(sent)             # 6. tokenize
        # 7. remove stopwords, and single leftover letters (like "g" from "4G")
        tokens = [w for w in tokens if w not in stop_words and len(w) > 1]
        if tokens:                               # skip sentences that became empty
            all_sentences.append(tokens)
    return all_sentences

print()
for i, text in enumerate(complaints, 1):
    print(f"Complaint {i} tokens:")
    for j, tokens in enumerate(preprocess(text), 1):
        print(f"  Sentence {j}:", tokens)


Complaint 1 tokens:
  Sentence 1: ['internet', 'not', 'working', 'since', 'days']
  Sentence 2: ['call', 'mail']
  Sentence 3: ['networkdown']
Complaint 2 tokens:
  Sentence 1: ['dr', 'sana']
  Sentence 2: ['bills', 'wrongly', 'charged', 'lahore']
  Sentence 3: ['nobody', 'listening', 'customers']
  Sentence 4: ['contact', 'billingissue']


**Stopword decision (the word "NOT"):**
The standard NLTK stopword list contains negation words such as *not*, *no* and *nor*. We **keep** them and remove all other stopwords. If "NOT" were removed, *"internet is NOT working"* would become *"internet working"*, which reverses the meaning of the complaint and would mislead any later sentiment or classification model.

## Q2 (c) Stemming vs Lemmatization
**Task:** Decide which technique suits Team A and which suits Team B, justify it in two lines, then apply `PorterStemmer` and `WordNetLemmatizer` to the word list and print a comparison table (use the correct POS tag where needed).

- **Team A:** search engine, where "connect" should also retrieve "connected", "connecting" and "connection", and speed matters most.
- **Team B:** monthly report generator, where the output must contain valid dictionary words (for example "was" becomes "be").

In [ ]:
# Team A (search engine): STEMMING. It is fast and cuts connected/connecting/
# connection down to "connect", so all of them are retrieved. The stem need not be a real word.
# Team B (report generator): LEMMATIZATION. It uses a dictionary, so the output is
# always a valid word (was -> be), which a report needs.

words = ["connected", "connecting", "charges", "was", "better", "studies"]

# correct POS tag for each word (v = verb, n = noun, a = adjective)
pos_tags = {"connected": "v", "connecting": "v", "charges": "n",
            "was": "v", "better": "a", "studies": "n"}

stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()

print()
print(f"{'Word':<12}{'Stem':<12}{'Lemma':<12}")
print("-" * 36)
for w in words:
    stem = stemmer.stem(w)
    lemma = lemmatizer.lemmatize(w, pos=pos_tags[w])
    print(f"{w:<12}{stem:<12}{lemma:<12}")


Word        Stem        Lemma       
------------------------------------


connected   connect     connect     
connecting  connect     connect     
charges     charg       charge      
was         wa          be          
better      better      good        
studies     studi       study       


**Justification:**

| Team | Technique | Reason |
|---|---|---|
| **Team A** (search engine) | **Stemming** | It is fast and rule-based, and it cuts *connected / connecting / connection* down to the same stem `connect`, so all of them are retrieved. The stem does not have to be a real word. |
| **Team B** (report generator) | **Lemmatization** | It uses a dictionary and POS tags, so the output is always a valid word (*was* becomes *be*, *better* becomes *good*, *studies* becomes *study*), which a readable report needs. |

## Q2 (d) Minimum Edit Distance
**Task:** Implement a dynamic-programming function `min_edit_distance(source, target)` with **insertion = 1**, **deletion = 1** and **substitution = 2**. Then write `correct_city(word, cities)` that returns the closest valid city, and test it on the typed words.

**Approach:**
- `D[i][j]` is the distance between the first *i* letters of `source` and the first *j* letters of `target`.
- The first row and column are filled with 0, 1, 2, ... (only insertions or deletions are needed).
- Each cell is the minimum of deletion, insertion and substitution (cost 0 if the letters match, otherwise 2).
- `correct_city` returns the city with the smallest distance.

In [ ]:
def min_edit_distance(source, target):
    n, m = len(source), len(target)
    # D[i][j] = distance between first i letters of source and first j letters of target
    D = [[0] * (m + 1) for _ in range(n + 1)]

    # turning a word into an empty string needs only deletions (and the reverse, insertions)
    for i in range(1, n + 1):
        D[i][0] = i
    for j in range(1, m + 1):
        D[0][j] = j

    for i in range(1, n + 1):
        for j in range(1, m + 1):
            # substitution costs 2, but 0 if the letters already match
            sub = 0 if source[i - 1] == target[j - 1] else 2
            D[i][j] = min(D[i - 1][j] + 1,        # deletion
                          D[i][j - 1] + 1,        # insertion
                          D[i - 1][j - 1] + sub)  # substitution / match
    return D[n][m]

def correct_city(word, cities):
    # pick the city with the smallest edit distance to the typed word
    return min(cities, key=lambda c: min_edit_distance(word.lower(), c.lower()))

cities = ["Islamabad", "Lahore", "Karachi", "Peshawar", "Quetta"]
typed = ["Islamabd", "Lahor", "Karchi"]

print()
for t in typed:
    best = correct_city(t, cities)
    dist = min_edit_distance(t.lower(), best.lower())
    print(t, "->", best, "| distance =", dist)


Islamabd -> Islamabad | distance = 1
Lahor -> Lahore | distance = 1
Karchi -> Karachi | distance = 1


---
# Question 3: Autocomplete and Sentiment Detection for a Food Delivery Startup
FoodExpress wants (i) a simple next-word suggestion feature for writing reviews and (ii) a sentiment classifier that flags unhappy customers.

### Setup: imports

In [ ]:
import math
import numpy as np
from collections import Counter
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

## Q3 (a) N-gram Language Model
**Task:** Without using any NLP library:
1. build unigram and bigram counts,
2. compute P(w<sub>i</sub> | w<sub>i-1</sub>) with add-one (Laplace) smoothing,
3. show that the unsmoothed P(late | food) is zero while the smoothed value is not,
4. compute the smoothed log-probability of *"the food was late"* and *"the delivery was late"* and state which one the model considers more likely and why.

**Note:** Vocabulary size **V** = all unique tokens including `</s>` but excluding `<s>`.

**Formulas:**
- Unsmoothed: P(w | prev) = count(prev, w) / count(prev)
- Add-one smoothed: P(w | prev) = (count(prev, w) + 1) / (count(prev) + V)

**Training corpus**

In [ ]:
corpus = [
    "<s> the food was delicious </s>", "<s> the delivery was late </s>",
    "<s> the food was cold </s>", "<s> the delivery was fast </s>",
    "<s> the delivery was late </s>"
]

**(i) Unigram and bigram counts**

In [ ]:
# (i) unigram and bigram counts
unigram = Counter()
bigram = Counter()
for sentence in corpus:
    tokens = sentence.split()
    unigram.update(tokens)                    # count single words
    bigram.update(zip(tokens, tokens[1:]))    # count pairs (previous word, next word)

# V = unique tokens including </s> but excluding <s>
V = len([w for w in unigram if w != "<s>"])
print("Vocabulary size V =", V)
print("Unigram counts:", dict(unigram))
print("Bigram counts:", {" ".join(k): c for k, c in bigram.items()})

Vocabulary size V = 9
Unigram counts: {'<s>': 5, 'the': 5, 'food': 2, 'was': 5, 'delicious': 1, '</s>': 5, 'delivery': 3, 'late': 2, 'cold': 1, 'fast': 1}
Bigram counts: {'<s> the': 5, 'the food': 2, 'food was': 2, 'was delicious': 1, 'delicious </s>': 1, 'the delivery': 3, 'delivery was': 3, 'was late': 2, 'late </s>': 2, 'was cold': 1, 'cold </s>': 1, 'was fast': 1, 'fast </s>': 1}


**(ii) Probabilities: unsmoothed (MLE) and add-one smoothed**

In [ ]:
# (ii) probabilities
def p_mle(prev, w):
    # unsmoothed: count(prev, w) / count(prev)
    return bigram[(prev, w)] / unigram[prev]

def p_smooth(prev, w):
    # add-one (Laplace): (count(prev, w) + 1) / (count(prev) + V)
    return (bigram[(prev, w)] + 1) / (unigram[prev] + V)

**(iii) P(late | food): before and after smoothing**

In [ ]:
# (iii) P(late | food) before and after smoothing
print("\nUnsmoothed P(late | food) =", p_mle("food", "late"))
print("Smoothed   P(late | food) = (0+1)/(%d+%d) = %.4f"
      % (unigram["food"], V, p_smooth("food", "late")))


Unsmoothed P(late | food) = 0.0
Smoothed   P(late | food) = (0+1)/(2+9) = 0.0909


**(iv) Smoothed log-probability of the two sentences**  (logs are added instead of multiplying small probabilities)

In [ ]:
# (iv) smoothed log-probability, shown word by word
def show_log_prob(sentence):
    tokens = ["<s>"] + sentence.split() + ["</s>"]
    total = 0.0
    print("\n'%s':" % sentence)
    for prev, w in zip(tokens, tokens[1:]):
        p = p_smooth(prev, w)
        total += math.log(p)                  # add logs instead of multiplying probabilities
        print("   P(%s | %s) = (%d+1)/(%d+%d) = %.4f   ln = %.4f"
              % (w, prev, bigram[(prev, w)], unigram[prev], V, p, math.log(p)))
    print("   Total log-probability = %.4f" % total)
    return total

lp1 = show_log_prob("the food was late")
lp2 = show_log_prob("the delivery was late")

print("\nMore likely:", "the delivery was late" if lp2 > lp1 else "the food was late")
# Reason: both sentences share the pairs <s> the, was late and late </s>.
# They differ only in "the food" + "food was" versus "the delivery" + "delivery was".
# "delivery" is more frequent after "the" (3 times vs 2 for "food"), and "was" always
# follows "delivery" (3 of 3 times), so the delivery sentence gets a higher probability.
print("Why: 'delivery' is more common than 'food' in the corpus (3 vs 2),")
print("so P(delivery | the) and P(was | delivery) are higher than P(food | the) and P(was | food).")


'the food was late':
   P(the | <s>) = (5+1)/(5+9) = 0.4286   ln = -0.8473
   P(food | the) = (2+1)/(5+9) = 0.2143   ln = -1.5404
   P(was | food) = (2+1)/(2+9) = 0.2727   ln = -1.2993
   P(late | was) = (2+1)/(5+9) = 0.2143   ln = -1.5404
   P(</s> | late) = (2+1)/(2+9) = 0.2727   ln = -1.2993
   Total log-probability = -6.5268

'the delivery was late':
   P(the | <s>) = (5+1)/(5+9) = 0.4286   ln = -0.8473
   P(delivery | the) = (3+1)/(5+9) = 0.2857   ln = -1.2528
   P(was | delivery) = (3+1)/(3+9) = 0.3333   ln = -1.0986
   P(late | was) = (2+1)/(5+9) = 0.2143   ln = -1.5404
   P(</s> | late) = (2+1)/(2+9) = 0.2727   ln = -1.2993
   Total log-probability = -6.0384

More likely: the delivery was late
Why: 'delivery' is more common than 'food' in the corpus (3 vs 2),
so P(delivery | the) and P(was | delivery) are higher than P(food | the) and P(was | food).


**Conclusion:** the model considers **"the delivery was late"** more likely.

Both sentences share the pairs `<s> the`, `was late` and `late </s>`. They differ only in *"the food" + "food was"* versus *"the delivery" + "delivery was"*. In the corpus, "delivery" follows "the" more often than "food" does (3 times vs 2), and "was" follows "delivery" in 3 out of 3 cases. So P(delivery | the) and P(was | delivery) are higher than P(food | the) and P(was | food), which gives the delivery sentence the higher log-probability.

## Q3 (b) Naive Bayes Classifier
**Task:** Convert the training reviews into bag-of-words features with `CountVectorizer` and train a `MultinomialNB` model (`alpha = 1.0`). Print the class priors and the predicted label with its probability for every test review.

**Approach:**
- The vectorizer is fitted on the **training data only**, then used to transform the test data.
- `alpha = 1.0` is add-one smoothing.
- The class priors are stored as logs, so `exp()` converts them back to probabilities.

In [ ]:
train = [("The food was delicious and fresh", "Positive"),
         ("Very fast delivery, loved it", "Positive"),
         ("Great taste and good packaging", "Positive"),
         ("Excellent service, will order again", "Positive"),
         ("The food was cold and stale", "Negative"),
         ("Delivery was very late", "Negative"),
         ("Terrible taste, waste of money", "Negative"),
         ("Worst service, rider was rude", "Negative")]
test = [("Fresh food and fast delivery", "Positive"),
        ("Late delivery and cold food", "Negative"),
        ("Excellent taste, will order again", "Positive"),
        ("Rude rider and terrible service", "Negative")]

# separate the texts and the labels
X_train_text = [t for t, label in train]
y_train = [label for t, label in train]
X_test_text = [t for t, label in test]
y_test = [label for t, label in test]

# bag-of-words: fit on training data only, then transform the test data
cv = CountVectorizer()
X_train_bow = cv.fit_transform(X_train_text)
X_test_bow = cv.transform(X_test_text)

nb = MultinomialNB(alpha=1.0)                 # alpha = 1.0 is add-one smoothing
nb.fit(X_train_bow, y_train)

# class priors are stored as logs, so exp() gives back the probability
print("\nClass priors:")
for c, lp in zip(nb.classes_, nb.class_log_prior_):
    print(" ", c, "->", round(math.exp(lp), 2))

nb_pred = nb.predict(X_test_bow)
nb_prob = nb.predict_proba(X_test_bow)
print("\nNaive Bayes predictions:")
for text, p, pr in zip(X_test_text, nb_pred, nb_prob):
    print("  %-35s -> %s (%.3f)" % (text, p, pr.max()))   # pr.max() = probability of predicted class


Class priors:
  Negative -> 0.5
  Positive -> 0.5

Naive Bayes predictions:
  Fresh food and fast delivery        -> Positive (0.845)
  Late delivery and cold food         -> Negative (0.746)
  Excellent taste, will order again   -> Positive (0.935)
  Rude rider and terrible service     -> Negative (0.855)


## Q3 (c) Logistic Regression and Comparison
**Task:** Using the same data, train a `LogisticRegression` model on TF-IDF features. Print its accuracy on the test set and the 3 most positive and 3 most negative words according to the learned coefficients. Compare its predictions with Naive Bayes and explain, in 2-3 lines, which model you would deploy given the very small training set.

**Reading the coefficients:** classes are sorted alphabetically (`Negative`, `Positive`), so a **positive** coefficient pushes towards *Positive* and a **negative** one towards *Negative*.

In [ ]:
# TF-IDF features (again fit on train only)
tfidf = TfidfVectorizer()
X_train_tf = tfidf.fit_transform(X_train_text)
X_test_tf = tfidf.transform(X_test_text)

lr = LogisticRegression()
lr.fit(X_train_tf, y_train)
lr_pred = lr.predict(X_test_tf)
lr_prob = lr.predict_proba(X_test_tf)

print("\nLogistic Regression accuracy:", accuracy_score(y_test, lr_pred))

# classes are sorted alphabetically (Negative, Positive), so a positive
# coefficient pushes towards "Positive" and a negative one towards "Negative"
words = tfidf.get_feature_names_out()
coefs = lr.coef_[0]
order = np.argsort(coefs)                     # indices from most negative to most positive

print("3 most positive words:", [(words[i], round(float(coefs[i]), 3)) for i in order[::-1][:3]])
print("3 most negative words:", [(words[i], round(float(coefs[i]), 3)) for i in order[:3]])

print("\nComparison of predictions (label and confidence):")
print("NB accuracy:", accuracy_score(y_test, nb_pred))
for text, true, a, pa, b, pb in zip(X_test_text, y_test, nb_pred, nb_prob, lr_pred, lr_prob):
    print("  %-35s true=%-8s NB=%s (%.2f)  LR=%s (%.2f)" % (text, true, a, pa.max(), b, pb.max()))

# Comparison: both models predict all 4 test reviews correctly (accuracy 1.0).
# I would deploy Naive Bayes. With only 8 training reviews, NB needs very few
# parameters and add-one smoothing handles unseen words, so it is stable on tiny data.
# LR is only about 55-59% confident on every review, because it has too little data
# to learn strong weights, while NB is much more confident (75-94%).
# LR's coefficients are also unreliable here (e.g. "was" appears as a negative word,
# which makes no sense). With more data, Logistic Regression would be a good choice.


Logistic Regression accuracy: 1.0
3 most positive words: [('delicious', 0.217), ('fresh', 0.217), ('loved', 0.205)]
3 most negative words: [('was', -0.288), ('late', -0.256), ('stale', -0.214)]

Comparison of predictions (label and confidence):
NB accuracy: 1.0
  Fresh food and fast delivery        true=Positive NB=Positive (0.84)  LR=Positive (0.57)
  Late delivery and cold food         true=Negative NB=Negative (0.75)  LR=Negative (0.54)
  Excellent taste, will order again   true=Positive NB=Positive (0.94)  LR=Positive (0.59)
  Rude rider and terrible service     true=Negative NB=Negative (0.85)  LR=Negative (0.56)


**Which model to deploy: Naive Bayes**
- Both models classify all 4 test reviews correctly (accuracy 1.0).
- With only 8 training reviews, Naive Bayes has very few parameters, and add-one smoothing handles unseen words, so it stays stable on tiny data.
- Logistic Regression is only about 55-59% confident on each review because it has too little data to learn strong weights, while Naive Bayes is much more confident (75-94%).
- Its coefficients are also unreliable here (for example "was" appears as a negative word, which makes no sense).
- With more training data, Logistic Regression would be a good choice.

---
**End of Notebook**